# Notebook 03: Local RF-DETR testing

Validate running the fine-tuned fly detector **locally** instead of Roboflow's
hosted API. Produces the same `[x1, y1, x2, y2, conf]` arrays the tracking pipeline
expects.

## Stages
1. Environment + config
2. Load model
3. Multi-frame inference + sanity checks (spread across the clip)
4. Visualization
5. Hosted comparison (optional)
6. Throughput estimate — run detection on the **first N frames** back-to-back,
   measure ms/frame, extrapolate how long a full uncached tracking run would take

Run cells top-to-bottom.

## 0. Install (terminal, once)

```bash
conda activate fly-tracking
conda install "pytorch>=2.4" torchvision cpuonly -c pytorch -y
pip install "rfdetr[plus]"
```

**Windows OpenMP crash (kernel dies at `import torch`):** mixed conda-forge + PyTorch
MKL loads two OpenMP runtimes. Before running Python in this session:

```powershell
$env:KMP_DUPLICATE_LIB_OK="TRUE"
```

Or in Python (must be **before** `import torch`):

```python
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
```

Long-term: use a clean env with PyTorch from `-c pytorch` only, or test on EPFL Linux
where this clash usually does not happen.

Restart the kernel after install.

## 1. Environment + config

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # before torch on Windows

import gc
import inspect
import sys
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import supervision as sv
import torch
from transformers.utils import is_torch_available

assert is_torch_available(), (
    f"Need torch >= 2.4 (have {torch.__version__}). See install cell."
)
from transformers import BackboneConfigMixin  # noqa: F401

sys.path.insert(0, "..")
from utils import load_config

REPO_ROOT = Path("..").resolve()
config = load_config(REPO_ROOT / "config.yaml")

WEIGHTS_PATH = REPO_ROOT / "RF-DETR_model" / "weights.pt"
RAW_VIDEO = REPO_ROOT / config.video.raw_path
THRESHOLD = float(config.tracker.detection_confidence_rfdetr)
CHECKPOINT_RESOLUTION = 640
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- run settings ---
# Fractions of clip length for spot-check frames (computed after we know n_total).
# e.g. [0, 0.25, 0.5, 0.9] on a 300-frame clip → frames 0, 74, 149, 269
FRAME_FRACTIONS = [0.0, 0.25, 0.5, 0.9]

# Throughput test: run predict() on this many consecutive frames from the start,
# then extrapolate runtime for the whole video (same pattern as run_tracking.py).
BENCHMARK_FRAMES = 30
WARMUP_RUNS = 1 if DEVICE == "cuda" else 0

print(f"torch {torch.__version__} | device {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
print(f"weights: {WEIGHTS_PATH}  exists={WEIGHTS_PATH.exists()}")
print(f"video:   {RAW_VIDEO}  exists={RAW_VIDEO.exists()}")
print(f"threshold={THRESHOLD}  resolution={CHECKPOINT_RESOLUTION}")

## 2. Checkpoint metadata

Quick read of training args before loading the full model.

In [ ]:
assert WEIGHTS_PATH.exists(), f"Missing weights: {WEIGHTS_PATH}"

meta = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=False)
args = meta.get("args")
args_dict = args.__dict__ if hasattr(args, "__dict__") else (args if isinstance(args, dict) else {})

print(f"keys: {sorted(meta.keys())}  |  size {WEIGHTS_PATH.stat().st_size / 1e6:.0f} MB")
for k in ("num_classes", "class_names", "resolution", "hidden_dim"):
    if k in args_dict:
        print(f"  {k}: {args_dict[k]}")

assert args_dict.get("num_classes") == 1
assert args_dict.get("class_names") == ["fly"]
assert args_dict.get("resolution") == CHECKPOINT_RESOLUTION

del meta, args, args_dict
gc.collect()
print("metadata OK")

## 3. Load model

Run once per kernel session.

In [ ]:
from rfdetr_plus import RFDETR2XLarge

gc.collect()
t0 = time.perf_counter()

# Roboflow checkpoint has no model_name in args — from_checkpoint cannot infer 2XL.
model = RFDETR2XLarge(
    pretrain_weights=str(WEIGHTS_PATH),
    accept_platform_model_license=True,
    resolution=CHECKPOINT_RESOLUTION,
)

if hasattr(model, "to"):
    model.to(DEVICE)
if hasattr(model, "eval"):
    model.eval()

print(f"Loaded in {time.perf_counter() - t0:.1f}s on {DEVICE}")
print(f"class_names: {getattr(model, 'class_names', 'n/a')}")
print(f"predict: {inspect.signature(model.predict)}")

## 4. Read test frames

In [ ]:
assert RAW_VIDEO.exists(), f"Video not found: {RAW_VIDEO}"

cap = cv2.VideoCapture(str(RAW_VIDEO))
if not cap.isOpened():
    raise ValueError(f"Cannot open: {RAW_VIDEO}")

n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
img_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
img_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
fps = cap.get(cv2.CAP_PROP_FPS) or config.video.fallback_fps

# Spread spot-check frames across the clip (works for 300-frame clips and longer).
frame_indices = sorted({
    min(int(round(f * (n_total - 1))), n_total - 1) for f in FRAME_FRACTIONS
})

frames: dict[int, np.ndarray] = {}
for idx in frame_indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, bgr = cap.read()
    if ok:
        frames[idx] = bgr
cap.release()

benchmark_n = min(BENCHMARK_FRAMES, n_total)

print(f"{img_w}x{img_h}, {n_total} frames @ {fps:.1f} fps")
print(f"spot-check frames: {frame_indices}")
print(f"throughput test will use first {benchmark_n} consecutive frames")

## 5. Local inference

In [ ]:
def detections_to_xyxyc(dets: sv.Detections) -> np.ndarray:
    if dets is None or len(dets) == 0 or dets.confidence is None:
        return np.empty((0, 5), dtype=np.float32)
    return np.hstack([dets.xyxy, dets.confidence[:, None]]).astype(np.float32)


local_results: dict[int, dict] = {}

with torch.inference_mode():
    for _ in range(WARMUP_RUNS):
        _ = model.predict(next(iter(frames.values())), threshold=THRESHOLD)

    for frame_idx, frame_bgr in frames.items():
        t0 = time.perf_counter()
        dets = model.predict(frame_bgr, threshold=THRESHOLD)
        ms = (time.perf_counter() - t0) * 1000
        arr = detections_to_xyxyc(dets)
        local_results[frame_idx] = {"dets": dets, "xyxyc": arr, "ms": ms}

        print(f"frame {frame_idx:>5}: {len(arr):>3} dets  {ms:>8.0f} ms")
        if len(arr):
            bad = (
                (arr[:, 0] < -1) | (arr[:, 1] < -1)
                | (arr[:, 2] > img_w + 1) | (arr[:, 3] > img_h + 1)
                | (arr[:, 2] <= arr[:, 0]) | (arr[:, 3] <= arr[:, 1])
            )
            assert not bad.any(), f"invalid boxes on frame {frame_idx}"

print("\nlocal inference OK")

## 6. Visualize

In [ ]:
show_idx = sorted(frames.keys())[0]
dets = local_results[show_idx]["dets"]

annotated = sv.BoxAnnotator(thickness=2).annotate(frames[show_idx].copy(), dets)
if len(dets) and dets.confidence is not None:
    labels = [f"{c:.2f}" for c in dets.confidence]
    annotated = sv.LabelAnnotator(text_scale=0.5).annotate(annotated, dets, labels)

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
ax.set_title(f"Local RF-DETR — frame {show_idx} ({len(dets)} boxes, thr={THRESHOLD})")
ax.axis("off")
plt.tight_layout()
plt.show()
plt.close(fig)

## 7. Hosted comparison

Skipped automatically if `creds_config.yaml` is missing or has no API key.

In [ ]:
CREDS_PATH = REPO_ROOT / "creds_config.yaml"
run_hosted = CREDS_PATH.exists()
if run_hosted:
    creds = load_config(CREDS_PATH)
    api_key = getattr(creds, "API_KEY", "") or ""
    model_id = getattr(creds, "MODEL_ID", "") or config.roboflow.model_id
    run_hosted = bool(api_key)

if not run_hosted:
    print("Skipping hosted comparison (no creds).")
else:
    from inference_sdk import InferenceHTTPClient
    from inference_sdk.http.entities import InferenceConfiguration

    client = InferenceHTTPClient(
        api_url=config.roboflow.inference_api_url,
        api_key=api_key,
    )
    client.configure(InferenceConfiguration(confidence_threshold=THRESHOLD))

    print(f"{'frame':>6}  {'local':>6}  {'hosted':>6}  {'local_ms':>9}  {'hosted_ms':>10}")
    print("-" * 48)
    for frame_idx, frame_bgr in frames.items():
        local_n = len(local_results[frame_idx]["xyxyc"])
        local_ms = local_results[frame_idx]["ms"]

        t0 = time.perf_counter()
        hosted = client.infer(frame_bgr, model_id=model_id)
        hosted_ms = (time.perf_counter() - t0) * 1000
        hosted_n = len(detections_to_xyxyc(sv.Detections.from_inference(hosted)))

        print(f"{frame_idx:>6}  {local_n:>6}  {hosted_n:>6}  {local_ms:>8.0f}  {hosted_ms:>9.0f}")

## 8. Throughput estimate

Runs `predict()` on the **first N consecutive frames** (N = min(30, video length)),
measures average ms/frame, then multiplies by total frame count to estimate how
long an uncached full tracking run would take. Same one-frame-at-a-time pattern as
`run_tracking.py` — not a batch GPU optimization test.

In [ ]:
cap = cv2.VideoCapture(str(RAW_VIDEO))
bench: list[np.ndarray] = []
while len(bench) < benchmark_n:
    ok, bgr = cap.read()
    if not ok:
        break
    bench.append(bgr)
cap.release()

with torch.inference_mode():
    for _ in range(WARMUP_RUNS):
        _ = model.predict(bench[0], threshold=THRESHOLD)

    t0 = time.perf_counter()
    counts = [len(model.predict(bgr, threshold=THRESHOLD)) for bgr in bench]
    elapsed = time.perf_counter() - t0

n = len(bench)
ms = (elapsed / n) * 1000
est_min = ms * n_total / 1000 / 60

print(f"Timed {n} consecutive frames (frames 0–{n - 1}) on {DEVICE}")
print(f"  {ms:.0f} ms/frame  ({1000/ms:.2f} fps)")
print(f"  dets/frame: min={min(counts)} max={max(counts)} mean={np.mean(counts):.1f}")
print(f"  extrapolated full video ({n_total} frames): ~{est_min:.1f} min")

del bench
gc.collect()

## 9. Next steps

If sections 5–8 pass, wire local inference into the pipeline
(`misc_software/ignore_mds/local_model_v2.md`).

If the kernel dies during **section 3 (load)**, check:
- `torch >= 2.4` and `is_torch_available()` is True
- no mixed pip/conda torch (`pip show torch` vs conda list pytorch)
- Windows OpenMP clash — run from a fresh terminal, not mixed envs